# Flood Risk Prediction — Ensemble
### Manuela Munoz Ramirez

Combines the three tabular models (Logistic Regression, Random Forest, XGBoost) into a
soft-voting ensemble that averages their predicted probabilities. The idea is the three
make different mistakes, so averaging them smooths out the errors.

Uses common.py like the rest, and loads the three models already trained in
backend/models.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import joblib
from pathlib import Path

import common

## 1. Load data and the three trained models

In [ ]:
df = common.load_data()
X, y = common.build_features(df)
X_train, X_test, y_train, y_test = common.chronological_split(X, y)

models_dir = Path("..") / "backend" / "models"
lr  = joblib.load(models_dir / "logistic_regression_real.joblib")
rf  = joblib.load(models_dir / "random_forest.joblib")
xgb = joblib.load(models_dir / "xgboost.joblib")

print("Loaded LR, RF and XGBoost from backend/models")

## 2. Build the ensemble

Soft voting: take each model's probability of high risk, average the three, threshold at
0.5 for the final call. Doing it by hand instead of sklearn's VotingClassifier because
the three were trained and saved separately, so this just averages their outputs without
retraining.

In [ ]:
p_lr  = lr.predict_proba(X_test)[:, 1]
p_rf  = rf.predict_proba(X_test)[:, 1]
p_xgb = xgb.predict_proba(X_test)[:, 1]

p_ensemble = (p_lr + p_rf + p_xgb) / 3
pred_ensemble = (p_ensemble >= 0.5).astype(int)

print("Ensemble probabilities ready")

## 3. Compare against the baseline

In [ ]:
results = [
    common.evaluate("Logistic Regression", y_test, lr.predict(X_test), p_lr),
    common.evaluate("Random Forest", y_test, rf.predict(X_test), p_rf),
    common.evaluate("XGBoost", y_test, xgb.predict(X_test), p_xgb),
    common.evaluate("Ensemble", y_test, pred_ensemble, p_ensemble),
    common.persistence_baseline(df, y_test),
]

common.comparison_table(results)

## 4. What this shows

The ensemble comes out ahead of all three individual models on F1 and MCC. Small gain but
consistent, which is the point of averaging: the three make different errors and the
average cancels some out.

So the ensemble is our best tabular result, and the one to serve as default in the
backend.

## 5. Save the ensemble

The ensemble isn't one trained object, it's the three models plus the averaging rule. I
save a small wrapper so the backend can load it and call predict like any other model.

In [ ]:
class SoftVotingEnsemble:
    """Averages the probability of the three tabular models."""
    def __init__(self, models):
        self.models = models

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


ensemble = SoftVotingEnsemble([lr, rf, xgb])

check = common.evaluate("Ensemble (check)", y_test,
                        ensemble.predict(X_test), ensemble.predict_proba(X_test)[:, 1])
print("Saved ensemble F1:", round(check["F1"], 3))

joblib.dump(ensemble, models_dir / "ensemble.joblib")
print("Saved to backend/models/ensemble.joblib")

## Notes for the report

Built a soft-voting ensemble averaging the probabilities of LR, RF and XGBoost. It beat
every individual model (F1 0.806 vs 0.802 for the best single one) and sits furthest
above the persistence baseline of 0.796. Saved to backend/models so the backend can serve
it as default. The LSTM was left out because it's a Keras model and doesn't load the same
way as the tabular ones.